# 1
### (a) Factorized Joint Distribution
The joint distribution factorizes as:

$$P(F,S,A,W)=P(F)\,P(W \mid F)\,P(S \mid F)\,P(A \mid F,S)$$

So each variable is conditioned only on its parents in the graph.

### (b) Meaning of Computing $P(F=\text{True} \mid A=\text{True})$
This means asking: "Given that the alarm has gone off, what is the probability that there is a fire?"

The evidence is $A=\text{True}$. The belief being updated is the system’s belief about whether there is a fire, which is modeled as the probability $P(F=\text{True})$. Before observing the alarm, the system has a prior belief about fire. After observing that the alarm is true, it updates this belief using Bayes’ rule to obtain the posterior probability:

$$P(F=\text{True} \mid A=\text{True})$$

### (c) Two Conditional Independence Relationships
Given whether there is a fire, the sprinkler and smoke are conditionally independent. So:
$$W \perp S \mid F$$

Also, given the fire status, the sprinkler provides no additional information about the alarm. So:
$$W \perp A \mid F$$

### (d) Number of Parameters
#### Full Joint Distribution
There are 4 binary variables, so the full joint table has $2^4 = 16$ entries. Since probabilities must sum to $1$, the number of independent parameters is $16 - 1 = 15$. So the full joint distribution needs $15 parameters$.

#### Bayesian Network Factorization
Using the factorization:
$$P(F,S,A,W)=P(F)P(W \mid F)P(S \mid F)P(A \mid F,S)$$

Count the parameters:
* $P(F)$ has 1 parameter
* $P(W \mid F)$ has 2 parameters, one for each value of $F$
* $P(S \mid F)$ has 2 parameters, one for each value of $F$
* $P(A \mid F,S)$ has 4 parameters, one for each combination of $F$ and $S$

So the total amount of parameters needed for the Bayesian Network is:
$$1 + 2 + 2 + 4 = 9 \text{ parameters}$$

#### Computational Implication
The Bayesian network requires fewer parameters than the full joint distribution since $9 < 15$. This makes the model more compact, easier to estimate from data, and more efficient for probabilistic inference. The savings become much larger as the number of variables increases.

### (e) Why a Probabilistic Model Is More Useful
A probabilistic model is useful because real-world smoke detection involves uncertainty. For example, an alarm may go off because of fire, smoke from cooking, sensor error, or other environmental conditions. Unlike a deterministic rule-based system, a probabilistic model can combine multiple pieces of evidence and assign degrees of belief to possible causes. This allows the system to make more flexible and intelligent decisions, such as estimating the probability of fire rather than simply declaring fire or no fire. This also leaves room for uncertainty, where a higher probability represents a stronger belief that there is a fire.

# 2
### (a) How Each Algorithm Computes the Posterior
#### Variable Elimination
Variable Elimination computes the posterior $P(R \mid W=\text{true})$ by directly manipulating the Bayesian network factors. For the Sprinkler network:
$$C \rightarrow R,\quad C \rightarrow S,\quad S \rightarrow W,\quad R \rightarrow W$$

The joint distribution factorizes as:
$$P(C,R,S,W)=P(C)P(R \mid C)P(S \mid C)P(W \mid S,R)$$

To compute:
$$P(R \mid W=\text{true})$$

Variable Elimination first restricts the factors using the evidence $W=\text{true}$. Then it eliminates variables that are not the query variable or evidence variable. For example, to compute $P(R \mid W=\text{true})$, it may eliminate $C$ and $S$ by multiplying relevant factors and summing them out. Finally, it normalizes the result so that the probabilities over $R$ sum to 1.

#### Junction Tree Inference
Junction Tree inference first transforms the Bayesian network into a tree of cliques, called a junction tree. Each clique contains a group of variables, and the factors from the original Bayesian network are assigned to appropriate cliques.

Evidence such as $W=\text{true}$ is incorporated into the clique potentials. Then the algorithm performs message passing between cliques, often called calibration. After calibration, each clique belief is consistent with the rest of the tree and contains correct marginal information for the variables in that clique.

To answer $P(R \mid W=\text{true})$, the algorithm finds a clique containing $R$, marginalizes that clique belief down to $R$, and normalizes if necessary.

### (b) Where Summation Occurs
#### Variable Elimination
Summation occurs explicitly in this step:
```
summed = sum_out(product, var)
```
This is where a hidden variable is eliminated.

#### Junction Tree
Summation occurs during message passing and during final marginalization.

When one clique sends a message to another clique, it sums out the variables that are in the sending clique but not in the separator between the two cliques. A typical message has the form:
$$m_{i \rightarrow j}(S_{ij}) = \sum_{C_i \setminus S_{ij}} \phi_i(C_i)$$

where $C_i$ is the sender clique and $S_{ij}$ is the separator between cliques.

Summation also occurs at the end when marginalizing a clique belief to get the query variable in this step:
```
return marginalize(clique.belief, query)
```

### (c) Where Multiplication Occurs
#### Variable Elimination
Multiplication occurs in this step:
```
product = multiply(related)
```

The algorithm multiplies all factors that involve the variable currently being eliminated.
Multiplication also occurs near the end:
```
result = normalize(multiply(factors))
```

After all hidden variables have been eliminated, the remaining factors are multiplied together to produce an unnormalized distribution over the query variable.

### Junction Tree
Multiplication occurs when constructing clique potentials and during message passing.

First, the original Bayesian network factors are multiplied into clique potentials. For example, a clique containing $(S,R,W)$ might receive the factor $P(W \mid S,R)$ and another clique containing $(C,S,R)$ might contain factors such as $P(C)$, $P(S \mid C)$, and $P(R \mid C)$. These factors are multiplied to form clique beliefs.

Multiplication also occurs when a clique receives messages from neighboring cliques. Its belief is updated by multiplying its local potential by incoming messages:
$$b_i(C_i)=\phi_i(C_i)\prod_{k \in N(i)} m_{k \rightarrow i}$$

So multiplication is used to combine local information with information received from other parts of the tree.

### (d) Why Variable Elimination (VE) Is Repeated For New Query But Not Junction Tree (JT)
Variable Elimination is query-specific. It chooses an elimination order based on the query and evidence, eliminates variables, and produces a result for that particular query. During this process, intermediate factors are created and then discarded. If the query changes, for example from $P(R \mid W=\text{true})$ to $P(S \mid W=\text{true})$, the algorithm usually has to run elimination again.

Junction Tree inference has a larger upfront cost because it builds and calibrates a global data structure. But after the junction tree is calibrated with the evidence, many different marginal queries can be answered efficiently by looking at the appropriate clique and marginalizing. So for the same evidence, JT does not need to redo the full inference process for every new query.

### (e) Computational Trade-Offs
#### Variable Elimination
Variable Elimination is often simpler and cheaper if you only need one or a few queries. It avoids building a full junction tree. Its cost depends heavily on the elimination order. A poor elimination order can create very large intermediate factors. The main cost comes from multiplying factors and summing out variables. The size of intermediate factors determines the computational difficulty.

#### Junction Tree
Junction Tree inference has a higher preprocessing cost. It must moralize the graph, triangulate it, form cliques, build the junction tree, assign factors, and calibrate the tree by message passing. However, once the tree is built and calibrated, answering multiple marginal queries is efficient. Like VE, its complexity depends on the size of the largest clique in the junction tree. Large cliques can make JT expensive in both time and memory.

#### Comparison
Both algorithms are exact inference methods, and both can become expensive when the graph has high treewidth. Variable Elimination performs inference for a specific query, while Junction Tree performs a more global inference procedure that prepares the model for many queries.

### (f) When VE is preferable and when JT is preferable
### VE preferable
Variable Elimination is preferable when you only need to answer a small number of queries. For example, if the smart system only needs one posterior, then VE is a good choice because it avoids the overhead of building and calibrating a junction tree.

### JT preferable
Junction Tree inference is preferable when many queries must be answered using the same evidence. For example, after observing $W=\text{true}$, suppose the system wants:
$$P(R \mid W=\text{true}),\quad P(S \mid W=\text{true}),\quad P(C \mid W=\text{true})$$

And maybe several pairwise marginals as well. In that case, building and calibrating the junction tree once, and paying the computational cost upfront, is useful because each later query can be answered more efficiently from the calibrated clique beliefs.

# 3

In [1]:
import random

def rejection_sampling(num_samples):
    kept_samples = 0
    kept_with_disease = 0

    for _ in range(num_samples):
        # Sample Disease from prior
        D = random.random() < 0.05

        # Sample Test conditioned on Disease
        if D:
            T_positive = random.random() < 0.9
        else:
            T_positive = random.random() < 0.1

        # Sample Symptom conditioned on Disease
        if D:
            S_true = random.random() < 0.8
        else:
            S_true = random.random() < 0.2

        # Keep only samples matching evidence: T = positive and S = true
        if T_positive and S_true:
            kept_samples += 1

            if D:
                kept_with_disease += 1

    # Estimate posterior
    if kept_samples == 0:
        return kept_samples, None

    posterior_estimate = kept_with_disease / kept_samples
    return kept_samples, posterior_estimate


sample_sizes = [100, 1000, 10000]

for n in sample_sizes:
    kept, estimate = rejection_sampling(n)
    print(f"Samples: {n}, Kept: {kept}, Estimated posterior: {estimate}")

Samples: 100, Kept: 6, Estimated posterior: 0.6666666666666666
Samples: 1000, Kept: 58, Estimated posterior: 0.7241379310344828
Samples: 10000, Kept: 550, Estimated posterior: 0.6781818181818182


| Number of Samples | Kept Samples | Estimated Posterior |
|------------------:|-------------:|--------------------:|
|               100 |            6 |              0.6667 |
|             1,000 |           58 |              0.7241 |
|            10,000 |          550 |              0.6782 |

As the number of kept samples increases, the probability estimate $\hat{P}$ is expected to get closer to the true probability $P$. However, even in the 10,000-sample run, only around $550$ samples are kept, so there can still be some sampling error. Because rejection sampling estimates the posterior only from the accepted samples, more kept samples generally reduce variance and make the estimate more reliable.

#### Why is this approximate inference rather than exact inference?
This is approximate inference because we are not computing the posterior by exactly summing over all possible variable assignments. Instead, we generate random samples and use the fraction of accepted samples to estimate the probability. The result depends on randomness, so it may be slightly different each time the simulation is run.

#### What happens to the variance as sample size increases?
As the sample size increases, the variance of the estimate decreases. With only 100 samples, the estimate may be far from the true posterior because very few samples match the evidence. With 10,000 samples, many more samples are kept, so the estimate tends to be more stable and closer to the true value.

#### Why does this approach scale better than exact inference in large networks?
Exact inference can become very expensive in large Bayesian networks because it may require summing over many possible combinations of variables. The number of possible assignments grows exponentially with the number of variables. Sampling methods avoid enumerating every possible assignment and instead estimate probabilities from generated samples, which can be more practical for large networks.

#### What is one limitation of rejection sampling when evidence is rare?
A major limitation is that if the evidence is rare, most generated samples will be rejected. For example, if $T=\text{positive}$ and $S=\text{true}$ occurred very rarely, then only a tiny fraction of samples would be kept. This makes rejection sampling inefficient because many samples are wasted, and the posterior estimate may have high variance unless we generate a very large number of samples.